<a href="https://colab.research.google.com/github/Sanchita3005/Cyber-security-/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -r requirements.txt
python app.py

In [ ]:
# app.py

from flask import Flask, request, render_template_string, redirect, session
import sqlite3
from werkzeug.security import generate_password_hash, check_password_hash
from datetime import datetime, timedelta

app = Flask(__name__)
app.secret_key = "secure-login-secret-key"

DB = "login.db"
MAX_ATTEMPTS = 5
LOCK_TIME = 5


def database():
    conn = sqlite3.connect(DB)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = database()

    conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL,
            failed_attempts INTEGER DEFAULT 0,
            locked_until TEXT
        )
    """)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT,
            status TEXT,
            time TEXT
        )
    """)

    user = conn.execute(
        "SELECT * FROM users WHERE username=?",
        ("student",)
    ).fetchone()

    if not user:
        conn.execute(
            "INSERT INTO users(username,password) VALUES(?,?)",
            ("student", generate_password_hash("Student@123"))
        )

    conn.commit()
    conn.close()


def log(username, status):
    conn = database()
    conn.execute(
        "INSERT INTO logs(username,status,time) VALUES(?,?,?)",
        (username, status, datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    )
    conn.commit()
    conn.close()


HTML = """
<!DOCTYPE html>
<html>
<head>
<title>Secure Login System</title>
<style>
body {
    font-family: Arial;
    background: #eef3f8;
}
.box {
    width: 350px;
    margin: 100px auto;
    padding: 30px;
    background: white;
    border-radius: 15px;
    box-shadow: 0 5px 20px #aaa;
}
input, button {
    width: 100%;
    padding: 12px;
    margin: 10px 0;
    box-sizing: border-box;
}
button {
    background: #1769e0;
    color: white;
    border: none;
    border-radius: 5px;
}
.error {
    color: red;
}
.success {
    color: green;
}
</style>
</head>

<body>
<div class="box">
{{ content|safe }}
</div>
</body>
</html>
"""


@app.route("/", methods=["GET", "POST"])
def login():

    message = ""

    if request.method == "POST":

        username = request.form.get("username", "").strip()
        password = request.form.get("password", "")

        if not username or not password:
            message = "<p class='error'>All fields are required.</p>"
            return render_template_string(
                HTML,
                content=login_form(message)
            )

        conn = database()

        user = conn.execute(
            "SELECT * FROM users WHERE username=?",
            (username,)
        ).fetchone()

        if not user:
            log(username, "FAILED")